### Weather 

In [2]:
# Import necessary libraries

import os
import time
import requests
import pandas as pd
from datetime import date, timedelta


In [3]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"

# Capitals
CAPITALS = [
    {"country": "Italy",   "country_code": "IT", "city": "Rome"},
    {"country": "Finland", "country_code": "FI", "city": "Helsinki"},
]

HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "cloud_cover",
]

MODEL = "era5"

today = date.today()
end_date = today - timedelta(days=6)
start_date = end_date.replace(year=end_date.year - 10)

OUT_DIR = "country_dfs"
os.makedirs(OUT_DIR, exist_ok=True)

def geocode(city: str, country_code: str) -> tuple[float, float, str]:
    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json",
        "countryCode": country_code,
    }
    r = requests.get(GEOCODE_URL, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    results = js.get("results") or []
    if not results:
        raise ValueError(f"No geocoding results for {city} ({country_code})")

    top = results[0]
    lat = float(top["latitude"])
    lon = float(top["longitude"])
    tz = top.get("timezone") or "auto"  # geocoding usually returns timezone
    return lat, lon, tz

def fetch_hourly(lat: float, lon: float, tz: str, d0: date, d1: date) -> pd.DataFrame:
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": d0.isoformat(),
        "end_date": d1.isoformat(),
        "hourly": ",".join(HOURLY_VARS),
        "timezone": tz,          # can also use "auto"
        "models": MODEL,
        "timeformat": "iso8601",
    }
    r = requests.get(ARCHIVE_URL, params=params, timeout=120)
    r.raise_for_status()
    js = r.json()

    hourly = js.get("hourly") or {}
    times = hourly.get("time") or []
    if not times:
        return pd.DataFrame()

    df = pd.DataFrame({"time": pd.to_datetime(times)})
    for v in HOURLY_VARS:
        if v in hourly:
            df[v] = hourly[v]

    df["latitude_used"] = js.get("latitude")
    df["longitude_used"] = js.get("longitude")
    df["timezone"] = js.get("timezone")
    return df

dfs = {}

print(f"Fetching hourly from {start_date} to {end_date} (model={MODEL})")

for item in CAPITALS:
    country = item["country"]
    cc = item["country_code"]
    city = item["city"]

    lat, lon, tz = geocode(city, cc)
    print(f"{country}: {city} -> ({lat:.4f}, {lon:.4f}) | tz={tz}")

    df = fetch_hourly(lat, lon, tz, start_date, end_date)

    df.insert(0, "country", country)
    df.insert(1, "country_code", cc)
    df.insert(2, "location", city)

    dfs[country.lower()] = df

    out_path = os.path.join(OUT_DIR, f"{country.lower()}_{city.lower()}_{start_date}_{end_date}_hourly.csv")
    df.to_csv(out_path, index=False)
    print(f"  saved: {out_path} | rows={len(df):,}")

    time.sleep(0.2)

df_italy = dfs["italy"]
df_finland = dfs["finland"]

print("\nDone. DataFrames available: df_italy, df_finland")

Fetching hourly from 2016-02-15 to 2026-02-15 (model=era5)
Italy: Rome -> (41.8919, 12.5113) | tz=Europe/Rome
  saved: country_dfs/italy_rome_2016-02-15_2026-02-15_hourly.csv | rows=87,696
Finland: Helsinki -> (60.1695, 24.9354) | tz=Europe/Helsinki
  saved: country_dfs/finland_helsinki_2016-02-15_2026-02-15_hourly.csv | rows=87,696

Done. DataFrames available: df_italy, df_finland
